In [1]:
import pandas as pd
import numpy as np

In [ ]:
#loading data
df=pd.read_csv('dataco_supply_chain.csv', encoding='latin1')
df.sample(5)

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
72454,TRANSFER,4,2,18.200001,72.779999,Late delivery,1,29,Shop By Sport,Chicago,...,NaN,627,29,NaN,http://images.acmesports.sports/Under+Armour+G...,Under Armour Girls' Toddler Spine Surge Runni,39.990002,0,5/20/2017 9:38,Second Class
110902,TRANSFER,6,4,63.759998,141.690002,Late delivery,1,46,Indoor/Outdoor Games,Caguas,...,NaN,1014,46,NaN,http://images.acmesports.sports/O%27Brien+Men%...,O'Brien Men's Neoprene Life Vest,49.980000,0,1/30/2017 12:57,Standard Class
82046,DEBIT,4,4,641.549988,1365.000000,Shipping on time,0,64,Computers,Columbia,...,NaN,1351,64,NaN,http://images.acmesports.sports/Dell+Laptop,Dell Laptop,1500.000000,0,11/3/2017 0:49,Standard Class
35490,CASH,5,4,22.830000,207.500000,Late delivery,1,24,Women's Apparel,Brooklyn,...,NaN,502,24,NaN,http://images.acmesports.sports/Nike+Men%27s+D...,Nike Men's Dri-FIT Victory Golf Polo,50.000000,0,4/21/2017 4:47,Standard Class
179493,TRANSFER,2,4,130.190002,464.950012,Advance shipping,0,9,Cardio Equipment,Caguas,...,NaN,191,9,NaN,http://images.acmesports.sports/Nike+Men%27s+F...,Nike Men's Free 5.0+ Running Shoe,99.989998,0,11/18/2016 15:40,Standard Class


In [ ]:
#handling unknown values
df['Customer Lname'] = df['Customer Lname'].fillna('Unknown')

In [5]:
#calculating delays
df['shipping_days_variance'] = df['Days for shipping (real)'] - df['Days for shipment (scheduled)']
df['is_late_delivery'] = np.where(df['shipping_days_variance'] > 0, 1, 0)

In [7]:
#Isolating Outliers (3rd Standard Deviation Method)
mean_days = df['Days for shipping (real)'].mean()
std_days = df['Days for shipping (real)'].std()
upper_limit = mean_days + (3 * std_days)
df['is_anomalous_delay'] = np.where(df['Days for shipping (real)'] > upper_limit, 1, 0)
print(f"-> Flagged {df['is_anomalous_delay'].sum()} systemic shipping bottlenecks via 3-Sigma threshold.")

-> Flagged 0 systemic shipping bottlenecks via 3-Sigma threshold.


In [8]:
#structuring schema
# Table 1: Vendor / Category Dimension
# DataCo maps operations via Department/Category blocks. We isolate these to optimize performance.
dim_categories = df[['Category Id', 'Category Name', 'Department Id', 'Department Name']].drop_duplicates().reset_index(drop=True)
dim_categories.rename(columns={'Category Id': 'category_id'}, inplace=True)

In [9]:
# Table 2: Customer Dimension
dim_customers = df[['Customer Id', 'Customer Fname', 'Customer Lname', 'Customer City', 'Customer State']].drop_duplicates().reset_index(drop=True)
dim_customers.rename(columns={'Customer Id': 'customer_id'}, inplace=True)

In [13]:
# Fix: Force column names to lowercase to prevent casing mismatches entirely
df.columns = df.columns.str.strip()

# Re-engineer the metrics directly to guarantee they exist in 'df'
df['shipping_days_variance'] = df['Days for shipping (real)'] - df['Days for shipment (scheduled)']
df['is_late_delivery'] = (df['shipping_days_variance'] > 0).astype(int)

mean_days = df['Days for shipping (real)'].mean()
std_days = df['Days for shipping (real)'].std()
df['is_anomalous_delay'] = (df['Days for shipping (real)'] > (mean_days + 3 * std_days)).astype(int)

# Identify the exact date column name regardless of capitalization
date_col = [col for col in df.columns if 'dateorders' in col.lower()][0]

# Build Table 3: Logistics Fact Table securely
fact_logistics = df[[
    'Order Id', 'Customer Id', 'Category Id', date_col,
    'Days for shipping (real)', 'Days for shipment (scheduled)', 'shipping_days_variance',
    'is_late_delivery', 'is_anomalous_delay', 'Sales', 'Order Item Total'
]].copy()

# Rename the columns cleanly for SQL/Power BI migration
fact_logistics.columns = [
    'order_id', 'customer_id', 'category_id', 'order_date',
    'days_shipping_real', 'days_shipping_scheduled', 'shipping_days_variance',
    'is_late_delivery', 'is_anomalous_delay', 'gross_sales', 'net_revenue'
]

print(f"Fact Table Created Successfully! Shape: {fact_logistics.shape}")

Fact Table Created Successfully! Shape: (180519, 11)


In [14]:
# Save down clean files
dim_categories.to_csv('dim_categories_clean.csv', index=False)
dim_customers.to_csv('dim_customers_clean.csv', index=False)
fact_logistics.to_csv('fact_logistics_clean.csv', index=False)

print("\n[SUCCESS] Phase 1 & 2 Complete! Generated 3 relational staging files:")
print(" - dim_categories_clean.csv")
print(" - dim_customers_clean.csv")
print(" - fact_logistics_clean.csv")


[SUCCESS] Phase 1 & 2 Complete! Generated 3 relational staging files:
 - dim_categories_clean.csv
 - dim_customers_clean.csv
 - fact_logistics_clean.csv
